# 211 — Post-hoc validation of clustering runs

Runs the three diagnostic analyses we agreed to add (Monti-style consensus,
Hennig per-cluster Jaccard, Tibshirani gap statistic, anatomical purity)
on every run currently in `outputs/clustering/index.json`.

Artifacts written into each run dir:
* `consensus_matrix.npy` (gitignored — large)
* `consensus_heatmap.png`
* `per_cluster_stability.csv`
* `stability_summary.json`
* `gap_by_k.json` + `gap_by_k.png`
* `per_cluster_anatomy.csv` + `per_cluster_anatomy.json`

MOBA's **Stats tab** reads these to show silhouette curve, gap curve,
per-cluster stability + anatomy purity per cluster.

**References:**
- Monti et al. 2003 — consensus clustering
- Hennig 2007 — per-cluster Jaccard stability
- Tibshirani et al. 2001 — gap statistic
- Hamilton et al. 2018 — anatomical-purity validation pattern for iEEG clustering


## 0. Imports + paths

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from functions.lf_stability  import save_consensus_artifacts
from functions.lf_anatomy    import build_aparc_cache, save_anatomy_artifacts
from functions               import lf_cluster_run as R

CLUSTERING_DIR = Path(R.DEFAULT_OUTPUTS_ROOT)
INDEX_PATH = CLUSTERING_DIR / 'index.json'

assert INDEX_PATH.exists(), f'No {INDEX_PATH} — run a clustering notebook first.'
with open(INDEX_PATH) as f:
    INDEX = json.load(f)
print(f'Found {len(INDEX["runs"])} runs in index.json:')
for r in INDEX['runs']:
    _s = r.get('silhouette'); _k = r.get('n_clusters')
    print(f"  {r['method']}/{r['feature_set']}/{r['run_id']}  K={_k}  "
          f"sil={_s:.3f}" if isinstance(_s, (int, float)) else
          f"  {r['method']}/{r['feature_set']}/{r['run_id']}  K={_k}  sil=n/a")



Found 39 runs in index.json:
  atlas/lana_inout/20260728_lana_p05_thr010  K=2  sil=n/a
  hierarchical/concat_hg/20260728_110823  K=6  sil=0.173
  hierarchical/concat_hg/20260731_215434  K=7  sil=0.166
  hierarchical/concat_raw/20260728_114228  K=4  sil=0.090
  hierarchical/concat_raw/20260731_224035  K=20  sil=0.087
  hierarchical/concat_rawds/20260728_110904  K=20  sil=0.099
  hierarchical/concat_rawds/20260731_215519  K=20  sil=0.104
  hierarchical/hg/20260605_183314  K=8  sil=0.117
  hierarchical/hg/20260719_153443  K=13  sil=0.110
  hierarchical/raw/20260605_185134  K=20  sil=0.039
  hierarchical/rawds/20260605_185323  K=16  sil=0.066
  hierarchical/rawds/20260719_155101  K=10  sil=0.069
  kmeans/concat_hg/20260728_110801  K=5  sil=0.201
  kmeans/concat_hg/20260731_215405  K=5  sil=0.195
  kmeans/concat_raw/20260728_110909  K=4  sil=0.096
  kmeans/concat_raw/20260731_215525  K=4  sil=0.096
  kmeans/concat_rawds/20260728_110829  K=16  sil=0.108
  kmeans/concat_rawds/20260731_215439 

## A. Consensus + per-cluster Jaccard stability

For each run, runs KMeans `N_RUNS` times at the run's best_k with different
random seeds. Each pair of samples gets a co-clustering frequency in [0, 1].
Per-cluster Jaccard = mean intra-cluster co-occurrence (excluding diagonal).
Stable cluster = high Jaccard (~0.9+); unstable = low (<0.5).

Cost: ~30s per KMeans run × N_RUNS per run-dir. For 8 runs × 50 consensus runs
this is ~3 hours total. Skip features you don't care about by editing `FEATURE_SETS`.


In [2]:
N_RUNS = 50         # consensus runs per cluster_run (more = more stable estimate)
FEATURE_SETS = ['hg', 'blob', 'minus101', 'raw', "rawds"]    # process in this order
SKIP_EXISTING = True

for run in INDEX['runs']:
    if run['feature_set'] not in FEATURE_SETS:
        continue
    run_dir = CLUSTERING_DIR / run['path']
    summary_path = run_dir / 'stability_summary.json'
    if SKIP_EXISTING and summary_path.exists():
        print(f'[skip] {run["path"]} (stability_summary.json exists)')
        continue
    X_path = run_dir / 'X_train.npy'
    if not X_path.exists():
        print(f'[skip] {run["path"]} (X_train.npy missing — rerun fit_and_save)')
        continue
    print(f'\n--- consensus: {run["path"]} (K={run["n_clusters"]}) ---')
    X = np.load(X_path)
    labels = pd.read_csv(run_dir / 'labels.csv')[f"cluster_{run['method']}_{run['feature_set']}"].to_numpy()
    save_consensus_artifacts(run_dir, X, labels, n_runs=N_RUNS)


[skip] hierarchical/hg/runs/20260605_183314 (stability_summary.json exists)
[skip] hierarchical/hg/runs/20260719_153443 (stability_summary.json exists)
[skip] hierarchical/raw/runs/20260605_185134 (stability_summary.json exists)
[skip] hierarchical/rawds/runs/20260605_185323 (stability_summary.json exists)
[skip] hierarchical/rawds/runs/20260719_155101 (stability_summary.json exists)
[skip] kmeans/hg/runs/20260605_183018 (stability_summary.json exists)
[skip] kmeans/hg/runs/20260605_183250 (stability_summary.json exists)
[skip] kmeans/hg/runs/20260719_153420 (stability_summary.json exists)
[skip] kmeans/minus101/runs/20260521_163821 (X_train.npy missing — rerun fit_and_save)
[skip] kmeans/minus101/runs/20260522_205443 (X_train.npy missing — rerun fit_and_save)
[skip] kmeans/minus101/runs/20260523_102735 (X_train.npy missing — rerun fit_and_save)
[skip] kmeans/raw/runs/20260521_151047 (X_train.npy missing — rerun fit_and_save)
[skip] kmeans/raw/runs/20260522_202441 (X_train.npy missing 

## E. Tibshirani gap statistic per run

For each run with `k_range` in the manifest, computes the gap statistic
across the whole K range. Writes `gap_by_k.json` + `gap_by_k.png` into the run dir.

Best K (Tibshirani rule) = smallest K such that Gap(K) >= Gap(K+1) − s_{K+1}.
Marked with a vertical line on the curve.

Cost: ~`N_REFS` extra KMeans fits per K × len(k_range). With N_REFS=10 and
K_RANGE=11 entries → ~110 extra KMeans per run. Slower for `raw`/`minus101`
(38k features); fast for `hg`/`blob`.


In [3]:
N_REFS = 10            # gap-stat null replicates per K
FEATURE_SETS_GAP = ['hg', 'blob']   # default: skip raw + minus101 (too slow at high-D)
SKIP_EXISTING = True

for run in INDEX['runs']:
    if run['feature_set'] not in FEATURE_SETS_GAP:
        continue
    run_dir = CLUSTERING_DIR / run['path']
    gap_path = run_dir / 'gap_by_k.json'
    if SKIP_EXISTING and gap_path.exists():
        print(f'[skip] {run["path"]} (gap_by_k.json exists)')
        continue
    manifest_path = run_dir / 'manifest.json'
    if not manifest_path.exists():
        continue
    manifest = json.loads(manifest_path.read_text())
    k_range = manifest.get('summary', {}).get('k_range')
    if not k_range:
        print(f'[skip] {run["path"]}: no k_range in manifest')
        continue
    X_path = run_dir / 'X_train.npy'
    if not X_path.exists():
        print(f'[skip] {run["path"]}: X_train.npy missing'); continue
    print(f'\n--- gap stat: {run["path"]} K_RANGE={k_range} ---')
    X = np.load(X_path)
    gap_by_k = R._compute_gap_statistic(X, k_range, n_refs=N_REFS, verbose=True)
    gap_path.write_text(json.dumps({str(k): v for k, v in gap_by_k.items()}, indent=2))
    R._save_gap_curve(run_dir / 'gap_by_k.png', gap_by_k,
                       title_prefix=f"{run['method']} · {run['feature_set']}")
    print(f'  -> {gap_path}')


[skip] hierarchical/hg/runs/20260605_183314 (gap_by_k.json exists)
[skip] hierarchical/hg/runs/20260719_153443 (gap_by_k.json exists)
[skip] kmeans/hg/runs/20260605_183018 (gap_by_k.json exists)
[skip] kmeans/hg/runs/20260605_183250 (gap_by_k.json exists)
[skip] kmeans/hg/runs/20260719_153420 (gap_by_k.json exists)


## C. Per-electrode anatomical labels (aparc) — one-time precompute

Builds a single cache CSV mapping (patient, electrode) -> Desikan-Killiany
aparc label, by projecting each electrode's MNI coord to the nearest fsaverage
cortical surface vertex. Requires `mne` and downloads fsaverage on first run
(~50MB, one-time). Cache lives at `02_FBM_Clustering/outputs/250_recon/fsaverage/aparc_lookup.csv`.


In [4]:
APARC_CACHE = CLUSTERING_DIR.parent / '250_recon' / 'fsaverage' / 'aparc_lookup.csv'
COORDS_GLOB = CLUSTERING_DIR.parent / '250_recon' / 'fsaverage' / 'coords' / '*_contacts_fsaverage.csv'

if APARC_CACHE.exists():
    print(f'[OK] aparc cache exists: {APARC_CACHE}')
    df_aparc = pd.read_csv(APARC_CACHE)
    print(f'  {len(df_aparc)} electrodes cached, {df_aparc["aparc_label"].nunique()} unique aparc labels')
    print(df_aparc['aparc_label'].value_counts().head(10))
else:
    print('Building aparc cache (this takes ~30s; mne fetches fsaverage on first run)...')
    df_aparc = build_aparc_cache(
        coords_csv_glob=str(COORDS_GLOB),
        output_csv=APARC_CACHE,
    )


[OK] aparc cache exists: \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon\fsaverage\aparc_lookup.csv
  3953 electrodes cached, 34 unique aparc labels
aparc_label
middletemporal          520
unknown                 406
insula                  364
rostralmiddlefrontal    360
inferiortemporal        315
superiortemporal        220
superiorfrontal         192
lateralorbitofrontal    153
fusiform                148
supramarginal           139
Name: count, dtype: int64


## C. Per-cluster anatomy purity per run

For each run, joins labels.csv (which has patient_id + electrode columns)
with the aparc cache. Computes per-cluster:
  * top 3 aparc regions + proportions
  * Shannon entropy (low = anatomically coherent)
  * purity = proportion of the modal region
  * permutation p-value (lower = more coherent than chance)


In [5]:
N_PERM = 500       # 0 to skip the permutation test (faster but no p-values)
SKIP_EXISTING = True

for run in INDEX['runs']:
    run_dir = CLUSTERING_DIR / run['path']
    out_csv = run_dir / 'per_cluster_anatomy.csv'
    if SKIP_EXISTING and out_csv.exists():
        print(f'[skip] {run["path"]} (per_cluster_anatomy.csv exists)')
        continue
    labels_path = run_dir / 'labels.csv'
    if not labels_path.exists():
        print(f'[skip] {run["path"]}: labels.csv missing'); continue
    df_l = pd.read_csv(labels_path)
    cluster_col = f"cluster_{run['method']}_{run['feature_set']}"
    if cluster_col not in df_l.columns:
        cands = [c for c in df_l.columns if c.startswith('cluster_')]
        if not cands: continue
        cluster_col = cands[0]
    if 'patient_id' not in df_l.columns or 'electrode' not in df_l.columns:
        print(f'[skip] {run["path"]}: labels.csv missing patient_id/electrode'); continue
    print(f'\n--- anatomy: {run["path"]} ---')
    save_anatomy_artifacts(run_dir, df_l, df_aparc,
                            cluster_col=cluster_col, n_perm=N_PERM)



--- anatomy: atlas/lana_inout/runs/20260728_lana_p05_thr010 ---
[anatomy] 2 clusters scored, wrote \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\atlas\lana_inout\runs\20260728_lana_p05_thr010\per_cluster_anatomy.csv
 cluster_id  n_total   purity  entropy_bits     top_region  top_proportion
          0     1602 0.164990      4.154626         insula           0.165
          1     1042 0.205212      4.085812 middletemporal           0.205
[skip] hierarchical/concat_hg/runs/20260728_110823 (per_cluster_anatomy.csv exists)

--- anatomy: hierarchical/concat_hg/runs/20260731_215434 ---
[anatomy] 7 clusters scored, wrote \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\hierarchical\concat_hg\runs\20260731_215434\per_cluster_anatomy.csv
 cluster_id  n_total   purity  entropy_bits       top_region  top_proportion
          0      122 0.178947      3.775405   middletemporal      

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\functions\lf_anatomy.py:444: RuntimeWarning: Mean of empty slice
  entry["null_entropy_mean"] = float(np.nanmean(null_entropies))



--- anatomy: hierarchical/concat_raw/runs/20260731_224035 ---
[anatomy] 20 clusters scored, wrote \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\hierarchical\concat_raw\runs\20260731_224035\per_cluster_anatomy.csv
 cluster_id  n_total   purity  entropy_bits           top_region  top_proportion
          0       65      NaN           NaN              unknown           1.000
          1       23      NaN           NaN              unknown           1.000
          2       31      NaN           NaN              unknown           1.000
          3       35 0.500000      1.000000        supramarginal           0.500
          4       36 0.264706      2.741916     inferiortemporal           0.265
          5      106 0.258065      2.687594               insula           0.258
          6       41 0.341463      2.362749 rostralmiddlefrontal           0.341
          7       59 0.339623      2.667957       middletemporal           0.3

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\functions\lf_anatomy.py:444: RuntimeWarning: Mean of empty slice
  entry["null_entropy_mean"] = float(np.nanmean(null_entropies))


[anatomy] 5 clusters scored, wrote \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\kmeans\concat_hg\runs\20260731_215405\per_cluster_anatomy.csv
 cluster_id  n_total   purity  entropy_bits       top_region  top_proportion
          0       83 0.282609      2.668764 superiortemporal           0.283
          1      167 0.291667      2.420670   middletemporal           0.292
          2      633 0.118343      4.302868   middletemporal           0.118
          3      118 0.177778      3.805293   middletemporal           0.178
          4      115 0.307692      3.180533   middletemporal           0.308
[skip] kmeans/concat_raw/runs/20260728_110909 (per_cluster_anatomy.csv exists)

--- anatomy: kmeans/concat_raw/runs/20260731_215525 ---
[anatomy] 4 clusters scored, wrote \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\kmeans\concat_raw\runs\20260731_215525\per_cluster_anatomy

\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\functions\lf_anatomy.py:444: RuntimeWarning: Mean of empty slice
  entry["null_entropy_mean"] = float(np.nanmean(null_entropies))



--- anatomy: kmeans/concat_rawds/runs/20260731_215439 ---
[anatomy] 19 clusters scored, wrote \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\kmeans\concat_rawds\runs\20260731_215439\per_cluster_anatomy.csv
 cluster_id  n_total   purity  entropy_bits           top_region  top_proportion
          0       93 0.217391      3.513629 rostralmiddlefrontal           0.217
          1       76 0.266667      2.840224     lateraloccipital           0.267
          2       70 0.187500      3.375220 rostralmiddlefrontal           0.188
          3      102 0.139785      3.993653       middletemporal           0.140
          4       46 0.322581      2.514659     inferiortemporal           0.323
          5       78 0.233333      2.957544       middletemporal           0.233
          6      107 0.291667      3.242829               insula           0.292
          7       34 0.800000      0.721928       middletemporal           0.800
     

## Done

Commit + push the new validation artifacts so MOBA picks them up:
```
git add 02_FBM_Clustering/outputs/clustering
git add 02_FBM_Clustering/outputs/250_recon/fsaverage/aparc_lookup.csv
git commit -m "Validation artifacts: consensus, gap stat, anatomy purity"
git push
```
